In [1]:
from pathlib import Path
import sys
from scipy.stats import ttest_ind

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Thesis code" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from Functions.data_utils import (
    plot_incremental_response_rate,
    uplift_by_decile_bin,
    coerce_metrics_to_numeric,
)

C:\Users\tsterk\AppData\Local\anaconda3\envs\causalml_py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to import duecredit due to No module named 'duecredit'


In [2]:
import pandas as pd
from pathlib import Path


def load_uplift_modeling_data(base_dir: str = "./Data/") -> pd.DataFrame:
    file_path = Path(base_dir) / "covariates_modeling_uplift_models_2026-03-13.csv"
    return pd.read_csv(file_path)


df = load_uplift_modeling_data()

C:\Users\tsterk\AppData\Local\Temp\ipykernel_10544\444953960.py:7: DtypeWarning: Columns (0: monetary_value, 1: total_volume, 2: online_sales, 3: retail_sales, 4: food_total, 5: vhms_total, 6: sports_total, 7: beauty_total, 8: monetary_value_52wk, 9: online_sales_52w, 10: retail_sales_52w, 11: monetary_value_53w_104w, 12: online_sales_53w_104w, 13: retail_sales_53w_104w) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file_path)


In [9]:
df['has_rfl'].value_counts(normalize = True)

has_rfl
1    0.851449
0    0.148551
Name: proportion, dtype: float64

In [4]:
df['treatment_indicator'].value_counts()

treatment_indicator
BNLX_ChurnP_10eu_controle_export.csv    15847
BNLX_ChurnP_10_controle_export.csv      15496
BNLX_ChurnP_250_test_export.csv         15412
BNLX_ChurnP_25_test_export.csv          15341
BNLX_ChurnP_500_controle_export.csv     15241
BNLX_ChurnP_25_controle_export.csv      15096
BNLX_ChurnP_500_test_export.csv         15019
BNLX_ChurnP_250_controle_export.csv     15013
BNLX_ChurnP_10eu_test_export.csv        15003
BNLX_ChurnP_5eu_controle_export.csv     14934
BNLX_ChurnP_5eu_test_export.csv         14830
BNLX_ChurnP_10_test_export.csv          14816
BNLX_ChurnP_SKUd_controle_export.csv    14740
BNLX_ChurnP_SKUe_test_export.csv        14675
BNLX_ChurnP_niks_controle_export.csv    14300
BNLX_ChurnP_SKUe_controle_export.csv    14184
BNLX_ChurnP_niks_test_export.csv        14117
BNLX_ChurnP_SKUd_test_export.csv        14079
Name: count, dtype: int64

In [5]:
# Extract offer and group from treatment_indicator
# e.g. "BNLX_ChurnP_10eu_controle_export.csv" -> offer="10eu", group="controle"
df["offer"] = df["treatment_indicator"].str.extract(r"BNLX_ChurnP_(.+?)_(test|controle)_export\.csv")[0]
df["group"] = df["treatment_indicator"].str.extract(r"BNLX_ChurnP_(.+?)_(test|controle)_export\.csv")[1]

def run_balance_ttests(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for offer, df_offer in df.groupby("offer"):
        df_test = df_offer[df_offer["group"] == "test"]
        df_control = df_offer[df_offer["group"] == "controle"]
        if len(df_test) == 0 or len(df_control) == 0:
            continue
        row = {"Offer": offer, "N Control": len(df_control), "N Treatment": len(df_test)}
        for metric in METRICS:
            stat, p_value = ttest_ind(
                df_test[metric].dropna(),
                df_control[metric].dropna(),
                equal_var=False
            )
            metric_clean = metric.replace("_", " ").title()
            row[f"{metric_clean} C"] = df_control[metric].mean()
            row[f"{metric_clean} T"] = df_test[metric].mean()
            row[f"{metric_clean} P-value"] = p_value
        results.append(row)
    return pd.DataFrame(results)

In [6]:
METRICS = ['recency', 'frequency', 'monetary_value']

df_clean = (
    df
    .pipe(coerce_metrics_to_numeric, cols=METRICS)
)

ttest_results = run_balance_ttests(df_clean)

ttest_results.to_excel("Output/ttest_results.xlsx", index=False)

In [7]:
ttest_results

,Offer,N Control,N Treatment,Recency C,Recency T,Recency P-value,Frequency C,Frequency T,Frequency P-value,Monetary Value C,Monetary Value T,Monetary Value P-value
0,10,15496,14816,921.391779,919.202079,0.627430,3.323309,3.283612,0.453567,73.120689,72.299584,0.563388
1,10eu,15847,15003,921.944090,920.166967,0.691264,3.275762,3.258015,0.715873,72.525467,72.349118,0.915102
2,25,15096,15341,917.025768,918.668796,0.714875,3.305578,3.313409,0.873977,71.974488,73.269566,0.319932
3,250,15013,15412,922.818424,911.699974,0.013742,3.353760,3.307488,0.406275,73.701600,72.539139,0.410976
4,500,15241,15019,915.944033,918.196751,0.619704,3.326619,3.285572,0.422206,73.117084,72.303627,0.542372
5,5eu,14934,14830,923.077206,918.285772,0.293085,3.319539,3.356440,0.540325,72.242757,72.670245,0.758865
6,SKUd,14740,14079,920.345590,918.540237,0.696944,3.262144,3.372399,0.036648,72.447693,73.230670,0.600870
7,SKUe,14184,14675,919.761492,915.030869,0.306774,3.311548,3.318501,0.896667,71.618212,72.444388,0.530333
8,niks,14300,14117,925.401958,917.789403,0.101576,3.268112,3.308564,0.454508,73.052737,72.327246,0.613891


In [8]:
from scipy.stats import chi2_contingency, fisher_exact
import pandas as pd

def gender_balance_by_offer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # extract offer and group from treatment_indicator
    parsed = df["treatment_indicator"].str.extract(r"BNLX_ChurnP_(.+?)_(test|controle)_export\.csv")
    df["offer"] = parsed[0]
    df["group"] = parsed[1].map({"controle": "control", "test": "test"})

    # clean gender
    df["gender_clean"] = df["gender"].fillna("Missing")

    def share(s: pd.Series, value: str) -> float:
        return (s == value).mean()

    results = []
    for offer, d in df.groupby("offer"):
        tab = pd.crosstab(d["group"], d["gender_clean"])
        if tab.shape[0] != 2:
            continue

        if (tab.values < 5).any() and tab.shape == (2, 2):
            stat, p = fisher_exact(tab.values)
            test = "fisher"
        else:
            stat, p, _, _ = chi2_contingency(tab)
            test = "chi2"

        d_control = d[d["group"] == "control"]
        d_test = d[d["group"] == "test"]

        results.append({
            "offer": offer,
            "female_control": share(d_control["gender_clean"], "F"),
            "male_control": share(d_control["gender_clean"], "M"),
            "missing_control": share(d_control["gender_clean"], "Missing"),
            "female_test": share(d_test["gender_clean"], "F"),
            "male_test": share(d_test["gender_clean"], "M"),
            "missing_test": share(d_test["gender_clean"], "Missing"),
            "p_value": p,
            "test_used": test,
        })

    return pd.DataFrame(results).sort_values("p_value", ascending=True)

gender_results = gender_balance_by_offer(df)
gender_results

,offer,female_control,male_control,missing_control,female_test,male_test,missing_test,p_value,test_used
6,SKUd,0.365739,0.082225,0.552035,0.371191,0.077278,0.551531,0.247535,chi2
2,25,0.366720,0.078431,0.554849,0.374617,0.077244,0.548139,0.360744,chi2
3,250,0.365283,0.080530,0.554186,0.368609,0.084025,0.547366,0.367796,chi2
4,500,0.373794,0.078997,0.547208,0.367867,0.079832,0.552300,0.565593,chi2
0,10,0.370418,0.081118,0.548464,0.365416,0.080454,0.554131,0.607248,chi2
1,10eu,0.363665,0.082665,0.553669,0.367793,0.080451,0.551756,0.646108,chi2
5,5eu,0.367283,0.082362,0.550355,0.370128,0.082063,0.547808,0.878378,chi2
8,niks,0.368252,0.080559,0.551189,0.367288,0.080187,0.552525,0.973747,chi2
7,SKUe,0.368514,0.083263,0.548223,0.368927,0.083543,0.547530,0.991795,chi2


In [9]:
gender_results.to_excel("Output/gender_results.xlsx", index=False)